# VQE-based Minimum Birkhoff Decomposition

> 🚀 **Don't choke your local machine.** Qoro is giving away **$100 in free cloud compute credits.**
> Get your API key at **[dash.qoroquantum.net](https://dash.qoroquantum.net)** to run this at scale.

## Why Cloud?

VQE optimizer iterations compound quickly — each iteration evaluates circuits, then the classical `black_box_optimizer` runs multi-threaded post-processing with caching. As matrix dimensions grow, the number of permutations explodes. QoroService offloads the **circuit evaluations** so the classical optimizer never waits for quantum results.

## Step 0 — Install & Authenticate

```bash
pip install qoro-divi pennylane
```

In [ ]:
import json

import numpy as np
import pennylane as qml

# Set your API key (get one free at https://dash.qoroquantum.net)

from birkhoff_vqe import BirkhoffDecomposition, combination_to_integer
from divi.backends import ParallelSimulator
from divi.qprog.algorithms._ansatze import GenericLayerAnsatz
from divi.qprog.optimizers import MonteCarloOptimizer, ScipyMethod, ScipyOptimizer

## How It Works

This example finds the **Birkhoff decomposition** of doubly stochastic matrices using VQE. It's built by **inheriting from Divi's VQE class** with only two changes:

1. **Override post-processing** — connects quantum measurements to the problem-specific `black_box_optimizer`
2. **Implement the classical routine** — multi-threaded optimizer with caching

All circuit execution, backend management, and optimization orchestration is inherited from Divi.

## Load Problem Instance

In [ ]:
# Configuration
N_DIM = 3            # Matrix dimension (3 or 4)
K_COMBINATIONS = 2   # Number of permutations in the decomposition
INSTANCE_ID = "1"    # Problem instance (1-10)
MATRIX_TYPE = "sparse"
MAX_ITERATIONS = 10

# Load problem data
dirname = os.path.dirname(os.path.abspath("__file__"))
mat_file = os.path.join(dirname, f"qbench_0{N_DIM}_{MATRIX_TYPE}.json")
perm_file = os.path.join(dirname, f"p{N_DIM}.dat")

with open(mat_file) as f:
    problem_data = json.load(f)
permutations = np.loadtxt(perm_file, dtype=int)

all_permutation_matrices = np.eye(N_DIM, dtype=int)[permutations - 1]

# Parse the specific instance
instance_data = problem_data[INSTANCE_ID]
matrix = np.array(instance_data["scaled_doubly_stochastic_matrix"]).reshape((N_DIM, N_DIM))
scale = instance_data["scale"]
sol_perms = np.eye(N_DIM, dtype=int)[np.array(instance_data["permutations"]).reshape(-1, N_DIM) - 1]
sol_weights = np.array(instance_data["weights"])

print(f"Matrix dimension: {N_DIM}×{N_DIM}")
print(f"Target decomposition: {K_COMBINATIONS} permutations")
print(f"Scale factor: {scale}")
print(f"\nTarget matrix (scaled):")
print(matrix)

---

## Run VQE

Uses Divi's **GenericLayerAnsatz** with RY rotations and CZ entanglers in a brick layout.

In [ ]:
optimizer = ScipyOptimizer(ScipyMethod.COBYLA)
qpu_backend = ParallelSimulator(shots=10000)
ansatz = GenericLayerAnsatz([qml.RY], entangler=qml.CZ, entangling_layout="brick")

birkhoff_vqe = BirkhoffDecomposition(
    matrix=matrix,
    scale=scale,
    n=N_DIM,
    k=K_COMBINATIONS,
    all_perms_matrix=all_permutation_matrices,
    backend=qpu_backend,
    optimizer=optimizer,
    max_iterations=MAX_ITERATIONS,
    ansatz=ansatz,
    n_layers=3,
)

print("Starting VQE optimization...")
birkhoff_vqe.run()

## Extract & Visualize Results

In [ ]:
combo, weights = birkhoff_vqe.find_final_decomposition()

if combo and weights:
    # Solution probability
    solution_integer = combination_to_integer(combo, birkhoff_vqe.k)
    solution_bitstring = format(solution_integer, f"0{birkhoff_vqe.n_qubits}b")
    probability = birkhoff_vqe.final_measurement_outcomes.get(solution_bitstring, 0) * 100

    print(f"Probability of Best Solution:")
    print(f"  Bitstring '{solution_bitstring}' — {probability:.2f}%")

    # Decomposition details
    print(f"\nVQE Decomposition:")
    for perm_idx, weight in zip(combo, weights):
        if weight > 1e-6:
            print(f"  Permutation {perm_idx}: weight = {weight:.6f}")

    # Reconstruction error
    vqe_reconstructed = sum(
        w * all_permutation_matrices[p] for p, w in zip(combo, weights) if w > 1e-6
    )
    original_unscaled = matrix / scale
    error_norm = np.linalg.norm(original_unscaled - vqe_reconstructed, ord=2)
    print(f"\nReconstruction error (L2): {error_norm:.6f}")
    print(f"\nOriginal (unscaled):\n{original_unscaled}")
    print(f"\nVQE reconstructed:\n{vqe_reconstructed}")
else:
    print("⚠️  No valid decomposition found. Try increasing MAX_ITERATIONS.")

---

## Ready for Larger Matrices?

Try `N_DIM=4` for a harder problem. When the VQE iterations get too heavy for local execution, switch to QoroService:

```python
from divi.backends import QoroService, JobConfig
qpu_backend = QoroService(job_config=JobConfig(shots=10_000))
```

👉 **[Get your free API key at dash.qoroquantum.net](https://dash.qoroquantum.net)** — $100 in credits, no credit card required.